# IMPORT LIBRARIES

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
from numba import njit
import math
import xarray as xr
import matplotlib.pyplot as plt
from google.colab import drive
from matplotlib.dates import DateFormatter
from scipy.optimize import minimize

##USE DATA FROM PYTHON

In [ ]:
!pip install aqua-fetch

In [ ]:
!pip install aqua-fetch netCDF4 xarray

In [ ]:

from aqua_fetch import RainfallRunoff

# Initialiser directement avec le nom du dataset
rr = RainfallRunoff("CAMELS_US")

dynamic data already exists as /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_US/camels_us_D.nc. To overwrite, set `overwrite=True`


In [ ]:
meta, ds = rr.fetch()

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. Use the EXACT variable names from your dataset
dynamic_features_to_keep = [
    "solrad_wm2",
    "airtemp_C_max",
    "airtemp_C_min",
    "dayl(s)",
    "vp_hpa",
    "pcp_mm",
    "q_cms_obs"
]

# 4. Subset the dataset
print("Extracting dynamic features...")
ds_optimized = ds.sel(dynamic_features=dynamic_features_to_keep)

# 5. Save the ultra-lightweight dataset to your Google Drive
save_path = '/content/drive/MyDrive/CAMELS_US_QC_Filtered.nc'
ds_optimized.to_netcdf(save_path)

print(f"✅ Success! Optimized dataset saved to {save_path}")

Extracting dynamic features...
✅ Success! Optimized dataset saved to /content/drive/MyDrive/CAMELS_US_QC_Filtered.nc


In [ ]:


# Loads instantly!
ds = xr.open_dataset('/content/drive/MyDrive/CAMELS_US_QC_Filtered.nc')

In [ ]:
print(ds.head())

<xarray.Dataset> Size: 134kB
Dimensions:           (time: 5, dynamic_features: 5)
Coordinates:
  * time              (time) datetime64[ns] 40B 1980-01-01 ... 1980-01-05
  * dynamic_features  (dynamic_features) <U13 260B 'solrad_wm2' ... 'vp_hpa'
Data variables: (12/671)
    02372250          (time, dynamic_features) float64 200B ...
    02077200          (time, dynamic_features) float64 200B ...
    02479300          (time, dynamic_features) float64 200B ...
    02481510          (time, dynamic_features) float64 200B ...
    02315500          (time, dynamic_features) float64 200B ...
    02231000          (time, dynamic_features) float64 200B ...
    ...                ...
    03364500          (time, dynamic_features) float64 200B ...
    03011800          (time, dynamic_features) float64 200B ...
    03066000          (time, dynamic_features) float64 200B ...
    03187500          (time, dynamic_features) float64 200B ...
    03366500          (time, dynamic_features) float64 200B ..

In [ ]:
print(ds.dynamic_features.values)

['solrad_wm2' 'airtemp_C_max' 'airtemp_C_min' 'dayl(s)' 'vp_hpa' 'pcp_mm'
 'q_cms_obs']


In [ ]:
# Full period
print(ds["time"].values[0], ds["time"].values[-1])

1980-01-01T00:00:00.000000000 2014-12-31T00:00:00.000000000


###General

In [ ]:
import xarray as xr
import numpy as np

# --- 1. Extraire les stations valides ---
station_keys = [k for k in ds.data_vars if len(k) == 8]

# --- 2. Convertir en DataArray avec dimension station_id ---
ds_stations = ds[station_keys].to_array(dim="station_id")  # dim='station_id' va contenir tous les noms

# --- 3. Extraire les débits observés (m³/s) ---
q_obs = ds_stations.sel(dynamic_features="q_cms_obs").drop_vars("dynamic_features").squeeze()
# Maintenant q_obs.dims = ('station_id', 'time')

# --- 4. Récupérer les surfaces de bassin (km²) ---
attrs = rr.fetch_static_features()
station_ids = q_obs.coords["station_id"].values
area = xr.DataArray(
    [attrs["area_km2"].get(sid, np.nan) for sid in station_ids],
    coords={"station_id": station_ids},
    dims=["station_id"]
)

# --- 5. Conversion m³/s → mm/jour ---
q_mm_per_day = (q_obs.where(q_obs >= 0) * 86.4) / area

print(q_mm_per_day)

<xarray.DataArray (station_id: 671, time: 12784)> Size: 69MB
array([[0.81180662, 0.77121629, 0.72635329, ...,        nan,        nan,
               nan],
       [0.24125942, 0.24125942, 0.22115446, ...,        nan,        nan,
               nan],
       [1.53953111, 1.45827808, 1.38343976, ..., 4.14818104, 3.76329827,
        3.03629746],
       ...,
       [1.73398589, 1.48627362, 1.26215299, ...,        nan,        nan,
               nan],
       [0.78073109, 0.66086781, 0.57340001, ...,        nan,        nan,
               nan],
       [0.72405579, 0.64601385, 0.58531456, ..., 0.32517476, 0.29699295,
        0.2666433 ]])
Coordinates:
  * station_id  (station_id) object 5kB '02372250' '02077200' ... '03300400'
  * time        (time) datetime64[ns] 102kB 1980-01-01 1980-01-02 ... 2014-12-31


**Functions**

In [ ]:
# ==========================================================
# 1. COMPILED MATH FUNCTION (FAO-56 Rigorous Net Radiation)
# ==========================================================
@njit(cache=True)
def compute_pt_pet(solrad, tmax, tmin, dayl, vp_pa, yday, lat_deg, elevation, alpha):
    """Calculates Priestley-Taylor PET strictly following Allen et al. (1998)."""
    N = len(solrad)
    PET = np.empty(N, dtype=np.float64)

    # Constants
    gamma = 0.066      # psychrometric constant (kPa C-1)
    lambda_v = 2.45    # latent heat of vaporization (MJ kg-1)
    albedo = 0.23      # standard reference albedo
    sigma = 4.903e-9   # Stefan-Boltzmann constant (MJ K-4 m-2 day-1)
    Gsc = 0.0820       # Solar constant (MJ m-2 min-1)

    # Latitude in radians
    lat_rad = lat_deg * np.pi / 180.0

    for i in range(N):
        T_mean = (tmax[i] + tmin[i]) * 0.5

        # --- 1. Vapor Pressures ---
        es_max = 0.6108 * np.exp((17.27 * tmax[i]) / (tmax[i] + 237.3))
        es_min = 0.6108 * np.exp((17.27 * tmin[i]) / (tmin[i] + 237.3))
        es = (es_max + es_min) * 0.5

        ea = vp_pa[i] / 10.0  # Convert Daymet Pa to kPa

        # --- 2. Radiation Balance (Rn) ---
        # Measured incoming shortwave (Rs) in MJ m-2 day-1
        Rs = solrad[i] * dayl[i] * 1e-6

        # Net shortwave (Rns)
        Rns = (1.0 - albedo) * Rs

        # Extraterrestrial Radiation (Ra) based on Julian Day (yday)
        J = yday[i]
        dr = 1.0 + 0.033 * np.cos(2.0 * np.pi * J / 365.0)
        delta = 0.409 * np.sin((2.0 * np.pi * J / 365.0) - 1.39)

        # Sunset hour angle (ws) with failsafe for high latitudes
        tan_lat_delta = np.tan(lat_rad) * np.tan(delta)
        if tan_lat_delta < -1.0:
            ws = 0.0
        elif tan_lat_delta > 1.0:
            ws = np.pi
        else:
            ws = np.arccos(-tan_lat_delta)

        Ra = (24.0 * 60.0 / np.pi) * Gsc * dr * (
            ws * np.sin(lat_rad) * np.sin(delta) +
            np.cos(lat_rad) * np.cos(delta) * np.sin(ws)
        )

        # Clear-sky solar radiation (Rso) adjusted for elevation
        Rso = (0.75 + 2e-5 * elevation) * Ra
        if Rso <= 0.0:
            Rso = 0.001 # Failsafe to prevent division by zero in winter

        # Cloudiness factor (f_cloud)
        Rs_Rso_ratio = Rs / Rso
        if Rs_Rso_ratio < 0.3: Rs_Rso_ratio = 0.3 # Bound per FAO-56
        if Rs_Rso_ratio > 1.0: Rs_Rso_ratio = 1.0
        f_cloud = 1.35 * Rs_Rso_ratio - 0.35

        # Net Longwave (Rnl)
        Tmax_K4 = (tmax[i] + 273.16)**4
        Tmin_K4 = (tmin[i] + 273.16)**4
        Rnl = sigma * ((Tmax_K4 + Tmin_K4) * 0.5) * (0.34 - 0.14 * np.sqrt(ea)) * f_cloud

        # Final Net Radiation (Rn)
        Rn = Rns - Rnl

        # --- 3. Priestley-Taylor PET ---
        # Slope of saturation vapor pressure curve at T_mean
        es_mean = 0.6108 * np.exp((17.27 * T_mean) / (T_mean + 237.3))
        s = 4098.0 * es_mean / ((T_mean + 237.3)**2)

        # Final calculation where Soil Heat Flux (G) = 0
        PET[i] = alpha * (s / (s + gamma)) * (Rn / lambda_v)

        # Failsafe for deep winter
        if PET[i] < 0.0:
            PET[i] = 0.0

    return PET

# ==========================================================
# 2. XARRAY DATA EXTRACTOR
# ==========================================================
def priestley_taylor_pet(ds_recent, station_id, alpha):
    """
    Extracts dynamic forcing and static basin data to call the compiled PET function.
    Requires alpha to be passed dynamically from your basin calibration loop.
    """
    station_data = ds_recent[station_id]

    # Dynamic Daymet Variables
    solrad = station_data.sel(dynamic_features="solrad_wm2").to_numpy()
    tmax = station_data.sel(dynamic_features="airtemp_C_max").to_numpy()
    tmin = station_data.sel(dynamic_features="airtemp_C_min").to_numpy()
    dayl = station_data.sel(dynamic_features="dayl(s)").to_numpy()
    vp_pa = station_data.sel(dynamic_features="vp_hpa").to_numpy()

    # Julian Day (Day of year 1-365) from the time index
    yday = station_data.time.dt.dayofyear.to_numpy()

    # Static Basin Variables (Update these accessors if your xarray metadata differs)
    # This assumes lat and elevation are stored as attributes or coordinates
    try:
        lat_deg = float(station_data.coords['lat'].values)
        elevation = float(station_data.coords['elevation'].values)
    except KeyError:
        # Fallback if they are stored in the dataset attributes instead
        lat_deg = float(station_data.attrs.get('latitude', 0.0))
        elevation = float(station_data.attrs.get('elevation', 0.0))

    # Execute the Numba-optimized math function
    return compute_pt_pet(solrad, tmax, tmin, dayl, vp_pa, yday, lat_deg, elevation, alpha)

## **Main Code** GR4J

In [ ]:
!pip install git+https://github.com/kratzert/RRMPG.git

  Cloning https://github.com/kratzert/RRMPG.git to /tmp/pip-req-build-582avo01
  Running command git clone --filter=blob:none --quiet https://github.com/kratzert/RRMPG.git /tmp/pip-req-build-582avo01
  Resolved https://github.com/kratzert/RRMPG.git to commit 7de78c25acc1c255d2acaf739d65e9ce7bbd60c3
  Preparing metadata (setup.py) ... done


In [ ]:
from rrmpg.models import GR4J

In [ ]:
# ============================================
# Metrics definitions
# ============================================

def NSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0 or np.var(obs) == 0:
        return np.nan
    return 1 - np.sum((sim - obs)**2) / np.sum((obs - np.mean(obs))**2)


def NNSE(nse):
    if np.isnan(nse):
        return np.nan
    return 1.0 / (2.0 - nse)


def RMSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    return np.sqrt(np.mean((sim - obs)**2))


def PBIAS(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0 or np.sum(obs) == 0:
        return np.nan
    return  np.sum(sim - obs) / np.sum(obs)


def FHV(obs, sim, top_fraction=0.02):
    epsilon = 0
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    n_top = int(len(obs) * top_fraction)
    if n_top == 0:
        return np.nan
    idx = np.argsort(obs)[-n_top:]
    obs_top = obs[idx]
    sim_top = sim[idx]
    return  np.sum(sim_top - obs_top) / (np.sum(obs_top) + epsilon)


def FLV(obs, sim, bottom_fraction=0.3):
    epsilon = 1e-6
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    n_bot = int(len(obs) * bottom_fraction)
    if n_bot == 0:
        return np.nan
    idx = np.argsort(obs)[:n_bot]
    obs_bot = obs[idx]
    sim_bot = sim[idx]
    return np.sum(sim_bot - obs_bot) / (np.sum(obs_bot) + epsilon)

In [ ]:
import numpy as np
from scipy.optimize import minimize

b1_ratio = 0.7
max_missing_ratio = 1
results_GR4J = {}

# station to skip
stations_to_skip = [""]

# Sélection des stations valides
stations = [s for s in q_mm_per_day.station_id.values if s not in stations_to_skip]

# periode
start_date = "1981-01-01"
end_date = "2013-12-31"

q_mm_per_day_recent = q_mm_per_day.sel(time=slice(start_date, end_date))
ds_recent = ds.sel(time=slice(start_date, end_date)).load()

# ======================
# NSE, NNSE, RMSE, PBIAS, FHV, FLV
# ======================

for i, station_id in enumerate(stations, 1):

    print(f"\n=== Station {station_id} ===, Number = {i}")

    # --- Observed runoff ---
    Q_obs = q_mm_per_day_recent.sel(station_id=station_id).to_numpy()

    # --- Forcing ---
    P = ds_recent[station_id].sel(dynamic_features="pcp_mm").to_numpy()
    base_PET = priestley_taylor_pet(ds_recent, station_id, alpha=1.0)

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        print("⚠️ Station skipped (no valid data).")
        continue

    missing_count = np.sum(np.isnan(Q_obs))
    missing_ratio = missing_count / N
    if missing_ratio > max_missing_ratio:
        print(f"⚠️ Too many missing values ({missing_ratio*100:.1f}%)")
        continue

    # --- split calibration / validation ---
    b1 = int(N * b1_ratio)
    Q_train = Q_obs[:b1]
    P_train = P[:b1]
    PET_train = base_PET[:b1]

    # --- Create model once ---
    model = GR4J()

    # --- objective function ---
    def objective_alpha(params, P_train, PET_train, Q_obs_train):
        x1, x2, x3, x4, alpha = params
        param_dict = dict(zip(model.get_parameter_names(), [x1, x2, x3, x4]))
        model.set_params(param_dict)

        PET_adj = alpha * PET_train
        try:
            Q_sim = model.simulate(P_train, PET_adj).flatten()
            nse = NSE(Q_obs_train, Q_sim)
            return 1 - nse if np.isfinite(nse) else 1e9
        except Exception:
            return 1e9

    # --- multi-start optimisation ---
    initial_guesses = [
        [350, 0, 90, 2, 1.5],
        [500, 1, 200, 3, 1.2],
        [200, -1, 100, 1.5, 1.8],
        [1000, 2, 300, 2.5, 1.0],
        [800, -2, 150, 1.8, 1.6]
    ]

    best_res = None
    best_val = float("inf")

    bounds = [
        (1, 3200),   # x1
        (-15, 15),   # x2
        (1, 1000),   # x3
        (0.5, 5),    # x4
        (None, None) # alpha libre
    ]

    for guess in initial_guesses:
        try:
            res = minimize(
                objective_alpha,
                guess,
                args=(P_train, PET_train, Q_train),
                method="L-BFGS-B",
                bounds=bounds,
                options={'maxiter': 2000, 'disp': False}
            )
            if res.fun < best_val:
                best_val = res.fun
                best_res = res
        except Exception as e:
            print(f"⚠️ Optimization failed for guess {guess}: {e}")
            continue

    if best_res is None:
        print("⚠️ No successful optimization for this station.")
        continue

    x1, x2, x3, x4, alpha_best = best_res.x

    # --- PET ajusted ---
    PET_full = alpha_best * base_PET

    # --- simulation completed ---
    param_dict = dict(zip(model.get_parameter_names(), [x1, x2, x3, x4]))
    model.set_params(param_dict)
    Qsim = model.simulate(P, PET_full).flatten()

    # --- Metrics Calibration ---
    mask_train = ~np.isnan(Q_train) & ~np.isnan(Qsim[:b1])
    Q_train_valid = Q_train[mask_train]
    Qsim_train_valid = Qsim[:b1][mask_train]

    NSE_cal = NSE(Q_train_valid, Qsim_train_valid)
    NNSE_cal = NNSE(NSE_cal)
    RMSE_cal = RMSE(Q_train_valid, Qsim_train_valid)
    PBIAS_cal = PBIAS(Q_train_valid, Qsim_train_valid)
    FHV_cal = FHV(Q_train_valid, Qsim_train_valid)
    FLV_cal = FLV(Q_train_valid, Qsim_train_valid)

    # --- Metrics Validation ---
    Q_val = Q_obs[b1:]
    Qsim_val = Qsim[b1:]
    mask_val = ~np.isnan(Q_val) & ~np.isnan(Qsim_val)
    Q_val_valid = Q_val[mask_val]
    Qsim_val_valid = Qsim_val[mask_val]

    NSE_val = NSE(Q_val_valid, Qsim_val_valid)
    NNSE_val = NNSE(NSE_val)
    RMSE_val = RMSE(Q_val_valid, Qsim_val_valid)
    PBIAS_val = PBIAS(Q_val_valid, Qsim_val_valid)
    FHV_val = FHV(Q_val_valid, Qsim_val_valid)
    FLV_val = FLV(Q_val_valid, Qsim_val_valid)

    # --- save ---
    results_GR4J[station_id] = {
        "params": [x1, x2, x3, x4, alpha_best],
        "NSE_cal": NSE_cal,
        "NNSE_cal": NNSE_cal,
        "RMSE_cal": RMSE_cal,
        "PBIAS_cal": PBIAS_cal,
        "FHV_cal": FHV_cal,
        "FLV_cal": FLV_cal,
        "NSE_val": NSE_val,
        "NNSE_val": NNSE_val,
        "RMSE_val": RMSE_val,
        "PBIAS_val": PBIAS_val,
        "FHV_val": FHV_val,
        "FLV_val": FLV_val,
        "Qsim": Qsim,
        "PET_adj": PET_full,
        "Q_obs": Q_obs,
        "missing_ratio": missing_ratio,
        "missing_count": missing_count,
    }

    print(f"✅ Station {station_id} metrics:")
    print(f"   Cal -> NSE: {NSE_cal:.3f}, NNSE: {NNSE_cal:.3f}, RMSE: {RMSE_cal:.3f}, PBIAS: {PBIAS_cal:.3f}, FHV: {FHV_cal:.3f}, FLV: {FLV_cal:.3f}")
    print(f"   Val -> NSE: {NSE_val:.3f}, NNSE: {NNSE_val:.3f}, RMSE: {RMSE_val:.3f}, PBIAS: {PBIAS_val:.3f}, FHV: {FHV_val:.3f}, FLV: {FLV_val:.3f}")
    print(f"   Params -> x1={x1:.2f}, x2={x2:.2f}, x3={x3:.2f}, x4={x4:.2f}, alpha={alpha_best:.2f}")

print(f"\n✅ Simulation complete for {len(results_GR4J)} valid basins.")


=== Station 02372250 ===, Number = 1


/tmp/ipykernel_27888/3959073871.py:93: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(
/tmp/ipykernel_27888/3959073871.py:93: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


✅ Station 02372250 metrics:
   Cal -> NSE: 0.750, NNSE: 0.800, RMSE: 1.108, PBIAS: 0.049, FHV: -0.314, FLV: 0.975
   Val -> NSE: 0.589, NNSE: 0.709, RMSE: 1.114, PBIAS: 0.281, FHV: -0.198, FLV: 1.916
   Params -> x1=293.46, x2=-11.63, x3=201.76, x4=1.37, alpha=0.20

=== Station 02077200 ===, Number = 2
✅ Station 02077200 metrics:
   Cal -> NSE: 0.549, NNSE: 0.689, RMSE: 2.213, PBIAS: 0.086, FHV: -0.358, FLV: 5.626
   Val -> NSE: 0.565, NNSE: 0.697, RMSE: 1.420, PBIAS: 0.233, FHV: -0.309, FLV: 6.288
   Params -> x1=430.06, x2=-4.94, x3=30.11, x4=0.50, alpha=0.16

=== Station 02479300 ===, Number = 3
✅ Station 02479300 metrics:
   Cal -> NSE: 0.742, NNSE: 0.795, RMSE: 1.436, PBIAS: 0.038, FHV: -0.225, FLV: 0.302
   Val -> NSE: 0.801, NNSE: 0.834, RMSE: 1.240, PBIAS: 0.072, FHV: -0.280, FLV: 0.425
   Params -> x1=614.38, x2=-4.18, x3=111.21, x4=2.37, alpha=0.22

=== Station 02481510 ===, Number = 4
✅ Station 02481510 metrics:
   Cal -> NSE: 0.771, NNSE: 0.814, RMSE: 1.592, PBIAS: 0.063, F

In [ ]:
# =============================================================
# 📌 FUNCTION TO PRINT METRIC SUMMARY
# =============================================================
def summarize_metric(metric_name, results_dict):

    values = [res.get(metric_name, np.nan) for res in results_dict.values()]
    values = np.array(values)
    values = values[~np.isnan(values)]  # remove NaN

    if len(values) == 0:
        print(f"{metric_name}: No valid values\n")
        return

    print(f"{metric_name}")
    print(f"Median : {np.percentile(values,50):.3f}")
    print(f"Mean   : {np.mean(values):.3f}")
    print(f"Min    : {np.min(values):.3f}")
    print(f"Max    : {np.max(values):.3f}")
    print(f"5th–95th percentile : {np.percentile(values,5):.3f} – {np.percentile(values,95):.3f}")
    print("-"*40)


# =============================================================
# 📌 CALIBRATION SUMMARY
# =============================================================
print("\n================= CALIBRATION SUMMARY =================\n")

metrics_cal = [
    "NSE_cal",
    "NNSE_cal",
    "RMSE_cal",
    "PBIAS_cal",
    "FHV_cal",
    "FLV_cal"
]

for metric in metrics_cal:
    summarize_metric(metric, results_GR4J)


# =============================================================
# 📌 VALIDATION SUMMARY
# =============================================================
print("\n================= VALIDATION SUMMARY =================\n")

metrics_val = [
    "NSE_val",
    "NNSE_val",
    "RMSE_val",
    "PBIAS_val",
    "FHV_val",
    "FLV_val"
]

for metric in metrics_val:
    summarize_metric(metric, results_GR4J)


================= CALIBRATION SUMMARY =================

NSE_cal
Median : 0.612
Mean   : 0.549
Min    : -0.621
Max    : 0.924
5th–95th percentile : 0.122 – 0.828
----------------------------------------
NNSE_cal
Median : 0.720
Mean   : 0.706
Min    : 0.381
Max    : 0.929
5th–95th percentile : 0.532 – 0.853
----------------------------------------
RMSE_cal
Median : 1.374
Mean   : 1.568
Min    : 0.044
Max    : 9.208
5th–95th percentile : 0.201 – 3.933
----------------------------------------
PBIAS_cal
Median : 0.002
Mean   : 0.018
Min    : -0.599
Max    : 1.165
5th–95th percentile : -0.143 – 0.231
----------------------------------------
FHV_cal
Median : -0.346
Mean   : -0.411
Min    : -0.892
Max    : -0.079
5th–95th percentile : -0.797 – -0.161
----------------------------------------
FLV_cal
Median : 1.451
Mean   : 4288650.659
Min    : -0.876
Max    : 576691594.685
5th–95th percentile : -0.296 – 414.939
----------------------------------------

================= VALIDATION SUMMARY ===